# Proyecto Despliegue de soluciones analíticas
**Predicción del rating de la pelicula en base de la trama**

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
from sentence_transformers import SentenceTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
dataTraining=pd.read_csv("dataTraining.csv")
dataTraining.head()

,Unnamed: 0,year,title,plot,genres,rating
0,3107,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0
1,900,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6
2,6724,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2
3,4704,1954,Executive Suite,"in a friday afternoon in new york , the presi...",['Drama'],7.4
4,2582,1990,Narrow Margin,"in los angeles , the editor of a publishing h...","['Action', 'Crime', 'Thriller']",6.6


In [3]:
dataTraining=dataTraining.drop(columns=['Unnamed: 0'])

In [4]:
dataTraining.head()

,year,title,plot,genres,rating
0,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0
1,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6
2,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2
3,1954,Executive Suite,"in a friday afternoon in new york , the presi...",['Drama'],7.4
4,1990,Narrow Margin,"in los angeles , the editor of a publishing h...","['Action', 'Crime', 'Thriller']",6.6


In [5]:
print("Valores nulos: \n", dataTraining[["plot", "rating", "genres"]].isnull().sum())

Valores nulos: 
 plot      0
rating    0
genres    0
dtype: int64


In [6]:
print("Registros duplicados: \n", dataTraining.duplicated().sum())

Registros duplicados: 
 1


In [7]:
dataTraining=dataTraining.drop_duplicates()

In [8]:
print("Estadisticas del rating:\n", dataTraining["rating"].describe())

Estadisticas del rating:
 count    7894.000000
mean        6.402876
std         1.078314
min         1.200000
25%         5.800000
50%         6.500000
75%         7.200000
max         9.300000
Name: rating, dtype: float64


In [9]:
x=dataTraining["plot"]
y=dataTraining["rating"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=0)

In [11]:
vector=TfidfVectorizer(stop_words='english', max_features=20000)

X_train_vectorizado=vector.fit_transform(X_train)
X_test_vectorizado=vector.transform(X_test)

In [12]:
modelo1=Ridge(alpha=1.0) #cambiar desde mlflow para ver si cambia el overfitting

modelo1.fit(X_train_vectorizado, y_train)
y_pred1=modelo1.predict(X_test_vectorizado)

In [13]:
mae=mean_absolute_error(y_test, y_pred1)
rmse=np.sqrt(mean_squared_error(y_test, y_pred1))
r2=r2_score(y_test, y_pred1)

print("mae:", round(mae, 4))
print("rmse:", round(rmse, 4))
print("R^2:", round(r2, 4))

mae: 0.7854
rmse: 1.0212
R^2: 0.1108


In [14]:
modelo2=SVR(kernel="linear", C=1.0)

modelo2.fit(X_train_vectorizado, y_train)
y_pred2=modelo2.predict(X_test_vectorizado)

In [15]:
mae2=mean_absolute_error(y_test, y_pred2)
rmse2=np.sqrt(mean_squared_error(y_test, y_pred2))
r22=r2_score(y_test, y_pred2)

print("mae:", round(mae2, 4))
print("rmse:", round(rmse2, 4))
print("R^2:", round(r22, 4))

mae: 0.7933
rmse: 1.0399
R^2: 0.0779


In [16]:
modelo3=SentenceTransformer("all-mpnet-base-v2")

In [17]:
X_train_embedding = modelo3.encode(X_train.tolist(), show_progress_bar=True)

X_test_embedding = modelo3.encode(X_test.tolist(), show_progress_bar=True)

print("Entrenamiento:", X_train_embedding.shape)
print("Validación:", X_test_embedding.shape)

Batches:   0%|          | 0/166 [00:00<?, ?it/s]

Batches:   0%|          | 0/82 [00:00<?, ?it/s]

Entrenamiento: (5288, 768)
Validación: (2606, 768)


In [19]:
np.save("X_train_embedding.npy", X_train_embedding)

In [20]:
X_test_embedding = modelo3.encode(X_test.tolist(), batch_size=8, show_progress_bar=False)

np.save("X_test_embedding.npy", X_test_embedding)
print("Validación:", X_test_embedding.shape)

Validación: (2606, 768)


In [22]:
modelo3_red = Sequential()

modelo3_red.add(Dense(192, activation="relu", input_shape=(768,)))
modelo3_red.add(Dropout(0.4))
modelo3_red.add(Dense(1, activation="linear"))

modelo3_red.compile(optimizer=Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
modelo3_red.summary()

c:\users\juanca mejía\appdata\local\programs\python\python39\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 192)                 │         147,648 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 192)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             193 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 147,841 (577.50 KB)

 Trainable params: 147,841 (577.50 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

historial = modelo3_red.fit(X_train_embedding, y_train, validation_split=0.2, epochs=100, batch_size=32, callbacks=[early_stop], verbose=1)

Epoch 1/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 18.5482 - mae: 3.6575 - val_loss: 2.2004 - val_mae: 1.1589
Epoch 2/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.4604 - mae: 1.2434 - val_loss: 1.7215 - val_mae: 1.0259
Epoch 3/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.9166 - mae: 1.0866 - val_loss: 1.5228 - val_mae: 0.9661
Epoch 4/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.7111 - mae: 1.0306 - val_loss: 1.4098 - val_mae: 0.9297
Epoch 5/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.4837 - mae: 0.9477 - val_loss: 1.3131 - val_mae: 0.8937
Epoch 6/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.4471 - mae: 0.9393 - val_loss: 1.2338 - val_mae: 0.8663
Epoch 7/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1.3387 - mae: 0.9072 - val_loss: 1.2034 - val_mae: 0.8562
Epoch 8/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.2997 - mae: 0.8894 - val_loss: 1.1706 - val_mae: 0.8387
Epoch 9/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms

In [25]:
y_pred3=modelo3_red.predict(X_test_embedding).flatten()

mae3= mean_absolute_error(y_test, y_pred3)
rmse3=np.sqrt(mean_squared_error(y_test, y_pred3))
r23=r2_score(y_test, y_pred3)

print("mae:", round(mae3, 4))
print("rmse:", round(rmse3, 4))
print("R²:", round(r23, 4))

82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
mae: 0.7736
rmse: 0.9961
R²: 0.1539


In [26]:
modelo3_red.save("modelo_rating_bert.keras")